# Data Transformation

## From Data Ingestion to Documents

In the LangChain pipeline, **data ingestion** is the first step: it takes raw files
(txt, PDF, HTML, JSON, CSV...) and converts them into **Document** objects.

A `Document` is the universal unit LangChain works with:

- `page_content`: the extracted text (a string)
- `metadata`: extra info about the source, like the file path (`source`)

Different loaders know how to read different formats, but they all return the same
`Document` wrapper. Right after ingestion comes **data transformation**: splitting
those big documents into smaller chunks with text splitters, so they fit the
token limits of LLMs and embedding models.

In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader

possible = [
    Path("data_ingestion"),
    Path("../data_ingestion"),
    Path.cwd() / "data_ingestion",
    Path.cwd().parent / "data_ingestion",
]
DATA_DIR = next((p for p in possible if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the data_ingestion folder")

text_loader = {
    "lyrics": DATA_DIR / "heylog_12_gauge_lyrics.txt",
    "paragraph": DATA_DIR / "random_paragraph.txt",
}

documents = []
for name, path in text_loader.items():
    loaded = TextLoader(str(path)).load()
    documents.extend(loaded)
    for doc in loaded:
        print(f"[{name}] source={doc.metadata['source']} | chars={len(doc.page_content)}")

print(f"Total documents loaded: {len(documents)}")
print("Example metadata:", documents[0].metadata)
print("Example content head:", documents[0].page_content[:40].replace(chr(10), " "))


[lyrics] source=../data_ingestion/heylog_12_gauge_lyrics.txt | chars=2735
[paragraph] source=../data_ingestion/random_paragraph.txt | chars=917
Total documents loaded: 2
Example metadata: {'source': '../data_ingestion/heylog_12_gauge_lyrics.txt'}
Example content head: 12 gauge - heylog (Album: eve, 2024)  [V


/tmp/ipykernel_91753/1366925207.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader



---
# Examples: Splitting the Loaded Documents

Now that files are `Document`s, we transform them into smaller chunks (data transformation). All splitters share the same API:
- `split_text(text)` -> `list[str]`
- `split_documents(docs)` -> `list[Document]` (keeps metadata)


## Example 1: RecursiveCharacterTextSplitter (default choice)

In [3]:
# unpack the two documents loaded earlier (keep the splitter cells short)
lyrics_doc, para_doc = documents


# Recursive: tries separators in order -> ["\n\n", "\n", " ", ...]
# so it only breaks mid-word as a last resort.
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter(
    chunk_size=120,      # max chars per chunk
    chunk_overlap=20,    # chars shared between neighbouring chunks
)

# split_text works on a plain string (from the loaded Document)
chunks = recursive.split_text(lyrics_doc.page_content)
print("num chunks:", len(chunks))
print("chunk sizes:", [len(c) for c in chunks])
print("\n--- chunk 0 ---\n", chunks[0])
print("\n--- chunk 1 ---\n", chunks[1])
print("\n--- chunk 2 ---\n", chunks[2])
print("\n--- chunk 3 ---\n", chunks[3])


num chunks: 31
chunk sizes: [36, 100, 82, 83, 92, 118, 60, 98, 43, 79, 92, 108, 102, 109, 77, 100, 90, 100, 119, 26, 83, 97, 83, 88, 117, 79, 89, 88, 107, 105, 47]

--- chunk 0 ---
 12 gauge - heylog
(Album: eve, 2024)

--- chunk 1 ---
 [Verse 1]
(Ooh)
Movin' swift, movin' clean
All my thrift big on me
And I look goofy when I see (Ooh)

--- chunk 2 ---
 Order double in XL when my body lean as hell
And it's all because I'm— (Well, ooh)

--- chunk 3 ---
 Maybe I should get clothes that I fit for my size
There are barely I own (Ooh, ooh)


## Example 2: CharacterTextSplitter (one fixed separator)

In [4]:

# Character: splits on ONE separator only, no fallback list.
# separator=" " -> words; separator="\n\n" -> paragraphs.
from langchain_text_splitters import CharacterTextSplitter

char_split = CharacterTextSplitter(
    separator=" ",       # only separator used
    chunk_size=200,      # target max chars
    chunk_overlap=20,
)

chunks = char_split.split_text(para_doc.page_content)
print("num chunks:", len(chunks))
print("chunk sizes:", [len(c) for c in chunks])
print("\n--- chunk 0 ---\n", chunks[0])


num chunks: 5
chunk sizes: [196, 195, 195, 197, 199]

--- chunk 0 ---
 The old library smelled of paper dust and warm wood, and every afternoon the same quiet man settled into the cracked leather chair by the window. He did not read so much as he inhabited the books,


## Example 3: HTMLHeaderTextSplitter (structure-aware)

In [5]:

# HTMLHeader: splits an HTML string by its header tags and stores
# the header hierarchy in each chunk's metadata.
from langchain_text_splitters import HTMLHeaderTextSplitter

html_string = """
<html><body>
<h1>Chapter 4</h1>
<p>Intro paragraph living under the h1.</p>
<h2>Method</h2>
<p>Method details under this h2.</p>
</body></html>
"""

html_split = HTMLHeaderTextSplitter(
    headers_to_split_on=[("h1", "Header 1"), ("h2", "Header 2")],
)
docs = html_split.split_text(html_string)

for i, d in enumerate(docs):
    print(f"[{i}] metadata = {d.metadata}")
    print("    content =", d.page_content.replace("\n", " "))


[0] metadata = {'Header 1': 'Chapter 4'}
    content = Chapter 4
[1] metadata = {'Header 1': 'Chapter 4'}
    content = Intro paragraph living under the h1.
[2] metadata = {'Header 1': 'Chapter 4', 'Header 2': 'Method'}
    content = Method
[3] metadata = {'Header 1': 'Chapter 4', 'Header 2': 'Method'}
    content = Method details under this h2.


## Example 4: RecursiveJsonSplitter (valid JSON chunks)

In [6]:

# RecursiveJson: splits a DICT (not a string!) by nesting depth so
# each chunk is still valid JSON under max_chunk_size.
# convert_lists=True lets it also chop big lists of dicts.
from langchain_text_splitters import RecursiveJsonSplitter
import json

payload = {
    "catalog": {"name": "my book store", "year": 2026},
    "books": [
        {"title": "To the Lighthouse", "author": "Virginia Woolf", "year": 1927},
        {"title": "The Left Hand of Darkness", "author": "Ursula K. Le Guin", "year": 1969},
        {"title": "Pedro Paramo", "author": "Juan Rulfo", "year": 1955},
        {"title": "The Sound and the Fury", "author": "William Faulkner", "year": 1929},
        {"title": "A Portrait of the Artist", "author": "James Joyce", "year": 1916},
    ],
}

json_split = RecursiveJsonSplitter(max_chunk_size=200, min_chunk_size=60)
chunks = json_split.split_text(payload, convert_lists=True)

print("num chunks:", len(chunks))
for i, c in enumerate(chunks):
    print(f"  chunk {i}: {len(c)} chars | valid JSON: "
          f"bool(json.loads(c)) | {c[:55]}")


num chunks: 3
  chunk 0: 142 chars | valid JSON: bool(json.loads(c)) | {"catalog": {"name": "my book store", "year": 2026}, "b
  chunk 1: 171 chars | valid JSON: bool(json.loads(c)) | {"books": {"1": {"title": "The Left Hand of Darkness", 
  chunk 2: 180 chars | valid JSON: bool(json.loads(c)) | {"books": {"3": {"title": "The Sound and the Fury", "au


## Example 5: split_documents (metadata travels with chunks)

In [7]:

# split_documents takes Document objects and returns chunked Documents,
# carrying each document's metadata (e.g. source file) along.
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc_split = RecursiveCharacterTextSplitter(
    chunk_size=150, chunk_overlap=25,
)
chunked_docs = doc_split.split_documents([lyrics_doc, para_doc])

print("total chunked documents:", len(chunked_docs))
for d in chunked_docs[:3]:
    print("- source:", d.metadata["source"], "| chars:", len(d.page_content))


total chunked documents: 33
- source: ../data_ingestion/heylog_12_gauge_lyrics.txt | chars: 36
- source: ../data_ingestion/heylog_12_gauge_lyrics.txt | chars: 145
- source: ../data_ingestion/heylog_12_gauge_lyrics.txt | chars: 121
